# Метрики по результатам tasks/variants/comparisons/judgments.csv

Учитывает человеческие оценоки (на будущее)

  * «кандидат» в эксперименте — первый вариант в названии: exp1 B (vs C), exp2 C (vs A), exp3 D (vs A), exp4 E (vs A), exp5 P (vs A). winrate_candidate - доля выборов кандидата.
  * invalid-вердикты исключаются из всех метрик (их доля в колонке invalid_rate)
  * Доверительные интервалы : кластерный бутстрэп по task_id (95%)

In [1]:
import argparse
import itertools
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats


In [2]:
EXP_ROLES = {  # experiment_type -> (кандидат, референс)
    "exp1_B_vs_C": ("B", "C"), "exp2_C_vs_A": ("C", "A"), "exp3_D_vs_A": ("D", "A"),
    "exp4_E_vs_A": ("E", "A"), "exp5_P_vs_A": ("P", "A"),
}
N_BOOT = 1000
SEED = 0


# ============================== Загрузка и подготовка ==============================
def load_tables(data_dir):
    d = Path(data_dir)
    rd = lambda n: pd.read_csv(d / n, dtype=object, keep_default_na=False)
    return rd("tasks.csv"), rd("variants.csv"), rd("comparisons.csv"), rd("judgments.csv")


def enrich_comparisons(comparisons, variants):
    """Добавляет к каждому сравнению: кандидат/референс, позицию кандидата, ключ пары, уровень длины."""
    vt = variants.set_index("variant_id")
    c = comparisons.copy()
    c["repeat_index"] = c.repeat_index.astype(int)
    type_left = c.variant_left_id.map(vt.variant_type)
    cand_type = c.experiment_type.map(lambda e: EXP_ROLES[e][0])
    cand_left = (type_left == cand_type).to_numpy()
    c["cand_id"] = np.where(cand_left, c.variant_left_id, c.variant_right_id)
    c["ref_id"] = np.where(cand_left, c.variant_right_id, c.variant_left_id)
    c["cand_first"] = cand_left.astype(int)                       # 1 — кандидат показан слева
    c["pair_key"] = ["|".join(sorted(p)) for p in zip(c.variant_left_id, c.variant_right_id)]
    c["cand_ratio"] = pd.to_numeric(c.cand_id.map(vt.target_length_ratio), errors="coerce")
    c["cand_actual"] = pd.to_numeric(c.cand_id.map(vt.actual_length_ratio), errors="coerce")
    c["source"] = c.cand_id.str.split("_").str[1]                 # smol / gemma / ...
    return c


def build_verdicts(meta, judgments):
    v = meta.merge(judgments[["comparison_id", "winner"]], on="comparison_id", how="inner")
    v["valid"] = v.winner.isin(["left", "right"])
    v["chose_left"] = np.where(v.valid, (v.winner == "left").astype(float), np.nan)
    v["chose_cand"] = np.where(v.cand_first == 1, v.chose_left, 1 - v.chose_left)  # NaN для invalid
    return v


# статистика
def cluster_mean_ci(df, col, n_boot=N_BOOT, seed=SEED):
    # Среднее по col и 95% CI (бутстрэп по task_id). Возвращает (est, lo, hi)
    d = df.dropna(subset=[col])
    if d.empty:
        return np.nan, np.nan, np.nan
    g = d.groupby("task_id")[col].agg(["sum", "count"])
    s, n = g["sum"].to_numpy(float), g["count"].to_numpy(float)
    idx = np.random.default_rng(seed).integers(0, len(g), (n_boot, len(g)))
    boots = s[idx].sum(1) / n[idx].sum(1)
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return s.sum() / n.sum(), lo, hi


def cohen_kappa(a, b):
    a, b = np.asarray(a), np.asarray(b)
    if len(a) == 0:
        return np.nan
    po = (a == b).mean()
    pe = sum((a == c).mean() * (b == c).mean() for c in np.union1d(a, b))
    return np.nan if pe >= 1 else (po - pe) / (1 - pe)


def _groups(d, by="experiment_type"):
    # ('ALL', все данные) + по группам by
    yield "ALL", d
    for k, g in d.groupby(by):
        yield k, g


def _safe_corr(fn, x, y):
    if len(x) < 3 or np.ptp(x) == 0 or np.ptp(y) == 0:
        return np.nan, np.nan
    r = fn(x, y)
    return float(r[0]), float(r[1])


#============================== 1. Win rate ==============================
def win_rates(v, extra=()):
    """Как часто судья выбирает кандидата в каждом эксперименте (опц. в разрезе extra: cand_ratio, source)."""
    rows = []
    for keys, d in v.groupby(["judge_id", "experiment_type", *extra]):
        judge, exp, *rest = keys
        est, lo, hi = cluster_mean_ci(d, "chose_cand")
        rows.append({"judge_id": judge, "experiment": exp, **dict(zip(extra, rest)),
                     "candidate": EXP_ROLES[exp][0], "reference": EXP_ROLES[exp][1],
                     "n_valid": int(d.valid.sum()), "invalid_rate": 1 - d.valid.mean(),
                     "winrate_candidate": est, "ci_low": lo, "ci_high": hi,
                     "winrate_reference": 1 - est})
    return pd.DataFrame(rows)


def _logit_fit(x, y, iters=30):
    X = np.column_stack([np.ones_like(x), x])
    b = np.zeros(2)
    for _ in range(iters):
        p = 1 / (1 + np.exp(-X @ b))
        w = p * (1 - p) + 1e-9
        step = np.linalg.solve(X.T @ (X * w[:, None]) + 1e-3 * np.eye(2), X.T @ (y - p) - 1e-3 * b)
        b = np.clip(b + step, -50, 50)
        if np.abs(step).max() < 1e-8:
            break
    return b


def length_slopes(v, n_boot=300):
    """Эксперимент 2: наклон логистической регрессии P(выбран C) ~ log(len(C)/len(A))"""
    d = v[(v.experiment_type == "exp2_C_vs_A") & v.valid & v.cand_actual.notna()]
    rows, rng = [], np.random.default_rng(SEED)
    for judge, g in d.groupby("judge_id"):
        x, y = np.log(g.cand_actual.to_numpy(float)), g.chose_cand.to_numpy(float)
        if np.ptp(x) == 0:
            continue
        b0, b1 = _logit_fit(x, y)
        tid = g.task_id.to_numpy()
        uniq = np.unique(tid)
        pos = {t: np.flatnonzero(tid == t) for t in uniq}
        boots = []
        for _ in range(n_boot):
            ii = np.concatenate([pos[t] for t in rng.choice(uniq, len(uniq))])
            boots.append(_logit_fit(x[ii], y[ii])[1])
        rows.append({"judge_id": judge, "n": len(g), "slope_logodds_per_logratio": b1,
                     "slope_ci_low": np.percentile(boots, 2.5), "slope_ci_high": np.percentile(boots, 97.5),
                     "p_choose_C_at_equal_length": 1 / (1 + np.exp(-b0))})
    return pd.DataFrame(rows)


# ============================== 2. Judge agreement и каппа между судьями ==============================
def item_probs(v):
    """P(судья выбирает кандидата) для каждой пары (по всем его вердиктам, в обоих порядках)"""
    d = v[v.valid]
    return (d.groupby(["experiment_type", "task_id", "pair_key", "judge_id"]).chose_cand.mean()
             .unstack("judge_id"))


def judge_agreement(p):
    """Вероятность, что случайный вердикт судьи 1 и случайный вердикт судьи 2 
    по случайной паре совпадут:
    для каждой пары p1*p2+(1-p1)*(1-p2),
    далее среднее по парам (среднее по всем комбинациям вердиктов)"""
    rows = []
    for j1, j2 in itertools.combinations(p.columns, 2):
        s = p[[j1, j2]].dropna().reset_index()
        s["agree"] = s[j1] * s[j2] + (1 - s[j1]) * (1 - s[j2])
        for exp, g in _groups(s):
            est, lo, hi = cluster_mean_ci(g, "agree")
            rows.append({"judge_1": j1, "judge_2": j2, "experiment": exp, "n_items": len(g),
                         "agreement": est, "ci_low": lo, "ci_high": hi})
    return pd.DataFrame(rows)


def kappa_between_judges(p):
    """Каппа Коэна между судьями на метках «кандидат/референс»; 
    таблица сопряжённости усреднена по всем комбинациям вердиктов в обоих порядках """
    rows = []
    for j1, j2 in itertools.combinations(p.columns, 2):
        s = p[[j1, j2]].dropna().reset_index()
        for exp, g in _groups(s):
            a, b = g[j1].to_numpy(), g[j2].to_numpy()
            T = np.array([[(a * b).mean(), (a * (1 - b)).mean()],
                          [((1 - a) * b).mean(), ((1 - a) * (1 - b)).mean()]])
            po, pe = T[0, 0] + T[1, 1], T.sum(1) @ T.sum(0)
            rows.append({"judge_1": j1, "judge_2": j2, "experiment": exp, "n_items": len(g),
                         "observed_agreement": po, "kappa": np.nan if pe >= 1 else (po - pe) / (1 - pe)})
    return pd.DataFrame(rows)


# ==============================  3. Корреляции с людьми, каппа ==============================
def load_human_prefs(meta, scores_path=None, pairs_path=None):
    """DataFrame[pair_key, human_pref]: >0 - человек считает кандидата лучше (для пар: +1/0/-1)"""
    pairs = meta.drop_duplicates("pair_key")[["pair_key", "cand_id", "ref_id"]]
    parts = []
    if scores_path and Path(scores_path).exists():
        sc = pd.read_csv(scores_path)
        m = sc.groupby("variant_id").score.mean()
        h = pairs.assign(human_pref=pairs.cand_id.map(m) - pairs.ref_id.map(m)).dropna(subset=["human_pref"])
        parts.append(h[["pair_key", "human_pref"]])
    if pairs_path and Path(pairs_path).exists():
        hp = pd.read_csv(pairs_path, dtype=str)
        hp["pair_key"] = ["|".join(sorted(p)) for p in zip(hp.variant_left_id, hp.variant_right_id)]
        hp["won_id"] = np.where(hp.human_winner == "left", hp.variant_left_id, hp.variant_right_id)
        hp = hp.merge(pairs, on="pair_key")
        hp["human_pref"] = np.where(hp.human_winner == "tie", 0.0, np.where(hp.won_id == hp.cand_id, 1.0, -1.0))
        parts.append(hp.groupby("pair_key", as_index=False).human_pref.mean())
    if not parts:
        return None
    return pd.concat(parts).drop_duplicates("pair_key", keep="last")


def human_correlations(v, human):
    """Pearson/Spearman/Kendall-tau между предпочтением судьи (среднее по вердиктам, [-1; 1]) и человека"""
    jp = (v[v.valid].groupby(["judge_id", "experiment_type", "pair_key"]).chose_cand.mean()
          .mul(2).sub(1).rename("judge_pref").reset_index().merge(human, on="pair_key"))
    rows = []
    for judge, dj in jp.groupby("judge_id"):
        for exp, g in _groups(dj):
            x, y = g.judge_pref.to_numpy(float), g.human_pref.to_numpy(float)
            (pr, pp), (sr, sp), (kt, kp) = (_safe_corr(stats.pearsonr, x, y),
                                            _safe_corr(stats.spearmanr, x, y),
                                            _safe_corr(stats.kendalltau, x, y))
            rows.append({"judge_id": judge, "experiment": exp, "n_pairs": len(g),
                         "pearson": pr, "pearson_p": pp, "spearman": sr, "spearman_p": sp,
                         "kendall_tau": kt, "kendall_p": kp})
    return pd.DataFrame(rows)


def kappa_vs_human(v, human):
    """Каппа Коэна: вердикт судьи (выбран кандидат?) против человека (кандидат лучше?); ничьи людей 
    должны исключаться"""
    d = v[v.valid].merge(human, on="pair_key")
    d = d[d.human_pref != 0]
    rows = []
    for judge, dj in d.groupby("judge_id"):
        for exp, g in _groups(dj):
            rows.append({"judge_id": judge, "experiment": exp, "n_verdicts": len(g),
                         "kappa": cohen_kappa(g.chose_cand.astype(int), (g.human_pref > 0).astype(int)),
                         "agreement": float((g.chose_cand.astype(int) == (g.human_pref > 0).astype(int)).mean())})
    return pd.DataFrame(rows)


# ============================== 4. Метрики ==============================
def position_metrics(v):
    """
    Position consistency: доля пар, где после перестановки ответов судья выбирает тот же ответ
    (для пары берутся все комбинации вердиктов кандидат слева * кандидат справа).
    Preference fairness/position bias (в статьях пишут разное): primacy_rate = доля выбора левого (1-го) ответа, recency_rate = правого.
    Каждая пара показывается в обоих порядках, для беспристрастного судьи должно быть primacy_rate = 0.5
    (position_bias = primacy_rate - 0.5; > 0 - primacy, < 0 - recency; fairness = 1 - 2|bias|).
    Несогласованные пары делятся на primacy-флип (выбран левый в обоих порядках) и recency-флип (правый в обоих).
    """
    d = v[v.valid]
    pp = (d.groupby(["judge_id", "experiment_type", "task_id", "pair_key", "cand_first"]).chose_cand.mean()
           .unstack("cand_first").dropna())
    pp.columns = ["p_cand_when_right", "p_cand_when_left"]       # cand_first = 0 / 1
    p0, p1 = pp.p_cand_when_right, pp.p_cand_when_left
    pp["consistent"] = p1 * p0 + (1 - p1) * (1 - p0)
    pp["primacy_flip"] = p1 * (1 - p0)       # левый в обоих порядках
    pp["recency_flip"] = (1 - p1) * p0       # правый в обоих порядках
    pp = pp.reset_index()
    rows = []
    for judge, dj in d.groupby("judge_id"):
        pj = pp[pp.judge_id == judge]
        for exp, g in _groups(dj):
            est, lo, hi = cluster_mean_ci(g, "chose_left")
            n, k = len(g), int(g.chose_left.sum())
            q = pj if exp == "ALL" else pj[pj.experiment_type == exp]
            flips = q.primacy_flip.sum() + q.recency_flip.sum()
            rows.append({
                "judge_id": judge, "experiment": exp, "n_verdicts": n, "n_pairs_both_orders": len(q),
                "primacy_rate": est, "recency_rate": 1 - est, "position_bias": est - 0.5,
                "bias_ci_low": lo - 0.5, "bias_ci_high": hi - 0.5,
                "bias_p_value": stats.binomtest(k, n, 0.5).pvalue if n else np.nan,
                "preference_fairness": 1 - 2 * abs(est - 0.5),
                "position_consistency": cluster_mean_ci(q, "consistent")[0],
                "consistency_ci_low": cluster_mean_ci(q, "consistent")[1],
                "consistency_ci_high": cluster_mean_ci(q, "consistent")[2],
                "primacy_flip_rate": q.primacy_flip.mean(), "recency_flip_rate": q.recency_flip.mean(),
                "primacy_share_of_flips": q.primacy_flip.sum() / flips if flips > 0 else np.nan})
    return pd.DataFrame(rows)


# ============================== 5.  Робастость ==============================
def robustness(wr, slopes, pos):
    """
    Робастость судьи по трем марам:
      placebo_robustness  = 1 - 2*|winrate(P vs A) - 0.5|  : судья не поддаётся переписыванию стиля (1 - идеально);
      placebo_consistency = position_consistency в exp5 - вердикт на плацебо не определяется порядком показа;
      control_detection   = доля случаев, когда исходный A побеждает E с потерей информации (~1);
      length_slope        = наклон по log-длине из эксперимента 2 (чем ближе к 0, тем устойчивее к длине);
    """
    rows = []
    for judge in wr.judge_id.unique():
        w = wr[wr.judge_id == judge].set_index("experiment")
        ps = pos[(pos.judge_id == judge) & (pos.experiment == "exp5_P_vs_A")]
        sl = slopes[slopes.judge_id == judge] if len(slopes) else slopes
        rows.append({
            "judge_id": judge,
            "placebo_winrate_P": w.winrate_candidate.get("exp5_P_vs_A", np.nan),
            "placebo_robustness": 1 - 2 * abs(w.winrate_candidate.get("exp5_P_vs_A", np.nan) - 0.5),
            "placebo_consistency": ps.position_consistency.iloc[0] if len(ps) else np.nan,
            "control_detection": 1 - w.winrate_candidate.get("exp4_E_vs_A", np.nan),
            "length_slope": sl.slope_logodds_per_logratio.iloc[0] if len(sl) else np.nan})
    return pd.DataFrame(rows)


# ============================== Метрики
def compute_all(data_dir="data", human_scores=None, human_pairs=None, out_dir=None, verbose=True):
    tasks, variants, comparisons, judgments = load_tables(data_dir)
    meta = enrich_comparisons(comparisons, variants)
    v = build_verdicts(meta, judgments)
    if v.empty:
        raise RuntimeError("judgments.csv пуст или не совпадает с comparisons.csv")
    p = item_probs(v)
    res = {"winrate": win_rates(v),
           "winrate_by_level": win_rates(v, extra=["cand_ratio"]),
           "winrate_by_source": win_rates(v, extra=["source"]),
           "length_slope": length_slopes(v),
           "judge_agreement": judge_agreement(p),
           "kappa_judges": kappa_between_judges(p),
           "position": position_metrics(v)}
    res["robustness"] = robustness(res["winrate"], res["length_slope"], res["position"])
    human = load_human_prefs(meta, human_scores, human_pairs)
    if human is not None and len(human):
        res["human_correlation"] = human_correlations(v, human)
        res["kappa_human"] = kappa_vs_human(v, human)
    elif verbose:
        print("Человеческие оценки не найдены: корреляции и каппа с людьми пропущены.")
    out = Path(out_dir) if out_dir else Path(data_dir) / "metrics"
    out.mkdir(parents=True, exist_ok=True)
    for name, df in res.items():
        df.to_csv(out / f"{name}.csv", index=False)
    if verbose:
        pd.set_option("display.width", 200, "display.max_columns", 30)
        print(f"Вердиктов: {len(v)} (invalid: {int((~v.valid).sum())}). Таблицы сохранены в {out}/")
        for name in ("winrate", "position", "robustness", "judge_agreement"):
            print(f"\n=== {name} ===")
            print(res[name].round(3).to_string(index=False))
    return res




In [3]:
DATA_DIR = "/kaggle/input/datasets/yumiasakura/sampled-dataset-v2"
OUT_DIR = "/kaggle/working/metrics" if Path("/kaggle/working").exists() else "metrics"
HUMAN_SCORES = None
HUMAN_PAIRS = None

res = compute_all(DATA_DIR, HUMAN_SCORES, HUMAN_PAIRS, OUT_DIR)

Человеческие оценки не найдены: корреляции и каппа с людьми пропущены.
Вердиктов: 675 (invalid: 11). Таблицы сохранены в /kaggle/working/metrics/

=== winrate ===
     judge_id  experiment candidate reference  n_valid  invalid_rate  winrate_candidate  ci_low  ci_high  winrate_reference
  judge_llama exp1_B_vs_C         B         C       37         0.000              0.459   0.368    0.571              0.541
  judge_llama exp2_C_vs_A         C         A       78         0.000              0.667   0.590    0.757              0.333
  judge_llama exp3_D_vs_A         D         A       58         0.000              0.517   0.424    0.633              0.483
  judge_llama exp4_E_vs_A         E         A       22         0.000              0.409   0.250    0.500              0.591
  judge_llama exp5_P_vs_A         P         A       30         0.000              0.533   0.500    0.600              0.467
judge_mistral exp1_B_vs_C         B         C       34         0.081              0.559   0.4

In [5]:
v = pd.read_csv("/kaggle/input/datasets/yumiasakura/sampled-dataset-v2/variants.csv"); d = v[v.variant_type != "A"].copy()
d["rel"] = d.actual_length_ratio.astype(float) / d.target_length_ratio.astype(float)
print(d.groupby(["variant_type", "target_length_ratio"]).rel.describe()[["mean", "min", "max"]])
d["preamble"] = d.text.str.lower().str.match(r"\s*(here|sure|certainly|rewritten|condensed)")
print(d.groupby("variant_type").preamble.mean())
print(pd.read_csv("/kaggle/input/datasets/yumiasakura/sampled-dataset-v2/judgments.csv").raw_response.str[:15].value_counts().head(10))

                                      mean       min       max
variant_type target_length_ratio                              
B            1.25                 0.894480  0.484800  1.176000
             2.00                 0.939050  0.525000  1.432000
C            1.25                 0.827040  0.412000  1.077600
             2.00                 0.851175  0.451000  1.437500
D            0.50                 0.909900  0.570000  1.594000
             0.75                 0.703400  0.394667  1.333333
E            0.75                 0.896667  0.582667  1.394667
P            1.00                 0.842700  0.509000  1.156000
variant_type
B    0.00
C    0.00
D    0.00
E    0.05
P    0.00
Name: preamble, dtype: float64
raw_response
{"lpA": -0.0, "    64
{"lpA": 0.0, "l    38
{"lpA": -0.0001    14
{"lpA": -0.0002     6
{"lpA": -0.0003     4
{"lpA": -0.0232     3
{"lpA": -0.0077     3
{"lpA": -0.0171     3
{"lpA": -0.0118     3
{"lpA": -0.5425     2
Name: count, dtype: int64


In [6]:
d[d.preamble].text.str[:80]

518    Here's a comprehensive guide on how to care fo...
Name: text, dtype: object